# AI of GOD 5.0 — Checkpoint Ensemble & Holdout Evaluation

Loads the saved five-fold `.pt` checkpoints for **ConvNeXt**, **DINO**, and **Swin**, runs each fold model on the untouched `holdout.npz` rows, averages probabilities across folds, and compares individual model families with an equal-weight ensemble.

**Important:** this notebook does not train models or tune ensemble weights on the holdout. Checkpoint architecture names and image sizes are configurable below; they must match the training notebooks exactly.


## 0. Configuration

In [ ]:

from pathlib import Path
import os, json, time, warnings
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import timm

SEED = 42
BASE = Path("/kaggle/working")
SPLITS_DIR = BASE / "splits"
OUT = BASE / "holdout_evaluation"
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
NUM_WORKERS = 2
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# These paths are searched recursively under /kaggle/input.
# Folder names follow the Kaggle dataset tree shown in your screenshots.
MODEL_CONFIGS = [
    {
        "name": "ConvNext Best",
        "folder_terms": ["convnext-best"],
        "checkpoint_glob": "fold*_best.pt",
        "timm_name": "convnext_base",
        "architecture": "convnext_custom",
        "hidden_dim": 512,
        "dropout": 0.3,
        "image_size": 224,
    },
    {
        "name": "DINO AoG best",
        "folder_terms": ["dino-aog-best"],
        "checkpoint_glob": "fold*_best.pt",
        # Register-token DINOv2 ViT-B/14 (4 register tokens). Must match training exactly.
        "timm_name": "dinov2_vitb14_reg",
        "architecture": "dino_custom",
        "hidden_dim": 512,
        "dropout": 0.3,
        "image_size": 280,
    },
    {
        "name": "Swin AOG best wts",
        "folder_terms": ["swin-aog-best-wts"],
        "checkpoint_glob": "fold*_best.pt",
        "timm_name": "swin_base_patch4_window7_224",
        "architecture": "swin_custom",
        "hidden_dim": 512,
        "dropout": 0.3,
        "image_size": 224,
    },
]
print("Device:", DEVICE)
print("timm:", timm.__version__)


## 1. Locate dataset, checkpoint files, and shared holdout split

In [ ]:

def find_one_file(filename, roots=("/kaggle/input", "/kaggle/working")):
    for root in roots:
        for dirpath, _, filenames in os.walk(root):
            if filename in filenames:
                return Path(dirpath) / filename
    raise FileNotFoundError(f"Could not find {filename} under {roots}")

train_csv = find_one_file("train.csv", roots=("/kaggle/input",))
train_df = pd.read_csv(train_csv)
required_cols = {"image_id", "animal_class"}
if not required_cols.issubset(train_df.columns):
    raise ValueError(f"train.csv must contain {required_cols}; found {list(train_df.columns)}")

classes = sorted(train_df["animal_class"].dropna().unique().tolist())
class_to_idx = {name: i for i, name in enumerate(classes)}
idx_to_class = {i: name for name, i in class_to_idx.items()}
train_df["label"] = train_df["animal_class"].map(class_to_idx).astype(int)

split_file = SPLITS_DIR / "holdout.npz"
if not split_file.exists():
    split_file = find_one_file("holdout.npz")
split = np.load(split_file)
holdout_idx = split["holdout_idx"].astype(int)
holdout_df = train_df.iloc[holdout_idx].reset_index(drop=True)
holdout_labels = holdout_df["label"].to_numpy()

print(f"CSV: {train_csv}")
print(f"Rows={len(train_df):,} | classes={len(classes)} | holdout={len(holdout_df):,}")
print(f"Holdout split: {split_file}")

# Resolve image paths robustly: try image_id as-is, then common image extensions.
image_roots = [Path("/kaggle/input")]
all_image_paths = {}
for root in image_roots:
    for ext in (".jpg", ".jpeg", ".png", ".webp", ".JPG", ".JPEG", ".PNG"):
        for p in root.rglob(f"*{ext}"):
            all_image_paths.setdefault(p.name, p)

def resolve_image(image_id):
    s = str(image_id)
    direct = Path(s)
    if direct.is_file():
        return direct
    if direct.name in all_image_paths:
        return all_image_paths[direct.name]
    if Path(s).suffix == "":
        for ext in (".jpg", ".jpeg", ".png", ".webp", ".JPG", ".JPEG", ".PNG"):
            if Path(s + ext).name in all_image_paths:
                return all_image_paths[Path(s + ext).name]
    raise FileNotFoundError(f"Image not found for image_id={image_id!r}")

holdout_df["image_path"] = holdout_df["image_id"].map(resolve_image)
print("Resolved holdout images:", holdout_df["image_path"].notna().sum())


## 2. Find the five checkpoints per model family

In [ ]:

import re

def find_checkpoint_group(cfg):
    """Find fold0_best.pt ... fold4_best.pt inside the configured Kaggle dataset."""
    terms = [t.lower() for t in cfg["folder_terms"]]
    found = []
    for p in Path("/kaggle/input/datasets/mayankmishra2006").rglob(cfg["checkpoint_glob"]):
        match = re.fullmatch(r"fold([0-4])_best\.pt", p.name, re.I)
        if not match:
            continue
        parent_chain = str(p.parent).lower()
        if any(term in parent_chain for term in terms):
            found.append(p)
    return sorted(found, key=lambda p: int(re.fullmatch(r"fold([0-4])_best\.pt", p.name, re.I).group(1)))

for cfg in MODEL_CONFIGS:
    cfg["checkpoints"] = find_checkpoint_group(cfg)
    print(f"\\n{cfg['name']}:")
    for p in cfg["checkpoints"]:
        print(" ", p)
    if len(cfg["checkpoints"]) != 5:
        raise FileNotFoundError(f"{cfg['name']}: expected 5 fold checkpoints, found {len(cfg['checkpoints'])}; terms={cfg['folder_terms']}")


## 3. Dataset and model/checkpoint helpers

### Architecture compatibility
ConvNeXt and Swin definitions mirror the uploaded training notebooks. DINOv2 is now matched to `05_dinov2_optuna_15_near_best`: official `torch.hub` `dinov2_vitb14_reg`, learnable `gem_p`, CLS + GeM pooled patch features, and the two-layer hidden MLP head. Input size is 280×280, matching training.


In [ ]:

class HoldoutDataset(Dataset):
    def __init__(self, frame, image_size):
        self.frame = frame.reset_index(drop=True)
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        with Image.open(row["image_path"]) as im:
            image = im.convert("RGB")
        return self.transform(image), int(row["label"])


# These model definitions mirror the uploaded training notebooks:
# ConvNeXt: torchvision ConvNeXt-Base + LayerNorm/Linear/GELU/Dropout/Linear head.
# Swin: timm Swin-Base with num_classes=0 + the same custom head.
class ConvNeXtClassifier(nn.Module):
    def __init__(self, num_classes, dropout=0.3, hidden_dim=512):
        super().__init__()
        from torchvision import models
        self.backbone = models.convnext_base(weights=None)
        in_f = self.backbone.classifier[2].in_features
        self.backbone.classifier = nn.Identity()
        self.head = nn.Sequential(
            nn.LayerNorm(in_f),
            nn.Linear(in_f, hidden_dim), nn.GELU(),
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, num_classes),
        )
    def forward(self, x):
        return self.head(self.backbone(x))


class SwinClassifier(nn.Module):
    def __init__(self, num_classes, dropout=0.3, hidden_dim=512):
        super().__init__()
        self.backbone = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=False, num_classes=0, global_pool="avg"
        )
        in_f = self.backbone.num_features
        self.head = nn.Sequential(
            nn.LayerNorm(in_f),
            nn.Linear(in_f, hidden_dim), nn.GELU(),
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, num_classes),
        )
    def forward(self, x):
        return self.head(self.backbone(x))


class DinoClassifier(nn.Module):
    """Exact architecture from 05_dinov2_optuna_15_near_best.ipynb."""
    def __init__(self, num_classes, model_name="dinov2_vitb14_reg",
                 dropout=0.32317124592062385, hidden_dim=512):
        super().__init__()
        # Use the same official backbone implementation as training.
        self.backbone = torch.hub.load(
            "facebookresearch/dinov2",
            model_name,
            pretrained=False,
        )
        ed = self.backbone.embed_dim  # 768 for ViT-Base
        self.gem_p = nn.Parameter(torch.ones(1) * 3.0)
        self.head = nn.Sequential(
            nn.LayerNorm(ed * 2),
            nn.Linear(ed * 2, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(p=dropout * 0.5),
            nn.Linear(hidden_dim // 2, num_classes),
        )

    def _gem_pool(self, patch_tokens):
        p = self.gem_p.clamp(min=1.0)
        return patch_tokens.clamp(min=1e-6).pow(p).mean(dim=1).pow(1.0 / p)

    def forward(self, x):
        feats = self.backbone.forward_features(x)
        cls = feats["x_norm_clstoken"]
        patch_tokens = feats["x_norm_patchtokens"]
        gem = self._gem_pool(patch_tokens)
        return self.head(torch.cat([cls, gem], dim=1))


def build_model(cfg):
    n = len(classes)
    if cfg["architecture"] == "convnext_custom":
        return ConvNeXtClassifier(n, cfg["dropout"], cfg["hidden_dim"])
    if cfg["architecture"] == "swin_custom":
        return SwinClassifier(n, cfg["dropout"], cfg["hidden_dim"])
    if cfg["architecture"] == "dino_custom":
        return DinoClassifier(n, cfg["timm_name"], cfg["dropout"], cfg["hidden_dim"])
    raise ValueError(f"Unknown architecture: {cfg['architecture']}")


def extract_state_dict(checkpoint):
    # Training notebooks save model.state_dict() directly; support wrappers too.
    if isinstance(checkpoint, dict):
        for key in ("state_dict", "model_state_dict", "model", "net", "weights", "model_ema", "ema"):
            if key in checkpoint and isinstance(checkpoint[key], dict):
                checkpoint = checkpoint[key]
                break
    if not isinstance(checkpoint, dict):
        raise TypeError("Checkpoint did not contain a state-dict-like mapping.")
    cleaned = {}
    for key, value in checkpoint.items():
        if not torch.is_tensor(value):
            continue
        k = key
        # Strip only common wrapper prefixes; preserve backbone./head. names.
        for prefix in ("module.", "_orig_mod."):
            if k.startswith(prefix):
                k = k[len(prefix):]
        cleaned[k] = value
    return cleaned

@torch.inference_mode()
def predict_checkpoint(cfg, checkpoint_path, frame):
    # Load weights first: Optuna may have selected hidden_dim=256/512/768.
    raw = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    state = extract_state_dict(raw)

    # Infer hidden width from the trained classifier instead of assuming 512.
    # ConvNeXt/Swin head: Linear at head.1, output layer at head.4
    # DINO head:          Linear at head.1, output layer at head.8
    cfg_local = dict(cfg)
    hidden_key = "head.1.weight"
    if hidden_key in state and state[hidden_key].ndim == 2:
        cfg_local["hidden_dim"] = int(state[hidden_key].shape[0])
        print(f"    inferred trained hidden_dim={cfg_local['hidden_dim']}")

    model = build_model(cfg_local)
    model_state = model.state_dict()
    model_keys, state_keys = set(model_state), set(state)
    matching = model_keys & state_keys
    shape_mismatch = [
        k for k in matching
        if tuple(model_state[k].shape) != tuple(state[k].shape)
    ]
    missing_keys = sorted(model_keys - state_keys)
    unexpected_keys = sorted(state_keys - model_keys)
    print(
        f"    checkpoint compatibility: matched={len(matching)}, "
        f"missing={len(missing_keys)}, unexpected={len(unexpected_keys)}, "
        f"shape_mismatch={len(shape_mismatch)}"
    )
    if shape_mismatch:
        print("    shape mismatch examples:", [
            (k, tuple(state[k].shape), tuple(model_state[k].shape))
            for k in shape_mismatch[:12]
        ])
    if missing_keys:
        print("    missing key examples:", missing_keys[:12])
    if unexpected_keys:
        print("    unexpected key examples:", unexpected_keys[:12])

    if missing_keys or unexpected_keys or shape_mismatch:
        raise RuntimeError(
            f"Checkpoint/model architecture mismatch for {checkpoint_path}. "
            "See the key/shape diagnostics above; do not evaluate partial weights."
        )

    model.load_state_dict(state, strict=True)
    model.to(DEVICE).eval()
    ds = HoldoutDataset(frame, cfg_local["image_size"])
    loader = DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda")
    )
    prob_batches = []
    start = time.time()
    for images, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        logits = model(images)
        prob_batches.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
    probs = np.concatenate(prob_batches, axis=0)
    elapsed = time.time() - start
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return probs, elapsed


## 4. Evaluate fold checkpoints and average their probabilities

In [ ]:

model_probabilities = {}
checkpoint_timing = []

for cfg in MODEL_CONFIGS:
    if not cfg["checkpoints"]:
        print(f"Skipping {cfg['name']}: no matching checkpoints found.")
        continue
    fold_probs = []
    print(f"\\n===== {cfg['name']} =====")
    for ckpt in cfg["checkpoints"]:
        print(f"Predicting {ckpt.name} ...")
        probs, seconds = predict_checkpoint(cfg, ckpt, holdout_df)
        fold_probs.append(probs)
        checkpoint_timing.append({
            "model": cfg["name"], "checkpoint": ckpt.name,
            "seconds": seconds, "seconds_per_image": seconds / len(holdout_df)
        })
    # Average the five fold models at probability level.
    family_probs = np.mean(np.stack(fold_probs, axis=0), axis=0)
    model_probabilities[cfg["name"]] = family_probs
    np.save(OUT / (cfg["name"].lower().replace(" ", "_") + "_holdout_probs.npy"), family_probs)
    print("Fold probabilities averaged:", family_probs.shape)

if not model_probabilities:
    raise RuntimeError("No checkpoints were evaluated. Check /kaggle/input paths and folder names.")


## 5. Holdout metrics: each model family and equal-weight ensemble

In [ ]:

def metric_row(name, probs, labels):
    preds = probs.argmax(axis=1)
    return {
        "model": name,
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
    }

rows = [metric_row(name, probs, holdout_labels)
        for name, probs in model_probabilities.items()]
ensemble_probs = np.mean(np.stack(list(model_probabilities.values()), axis=0), axis=0)
rows.append(metric_row("Equal-weight ensemble", ensemble_probs, holdout_labels))
metrics_df = pd.DataFrame(rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
display(metrics_df.style.format({"accuracy": "{:.4f}", "macro_f1": "{:.4f}", "weighted_f1": "{:.4f}"}))
metrics_df.to_csv(OUT / "holdout_metrics.csv", index=False)

timing_df = pd.DataFrame(checkpoint_timing)
timing_df.to_csv(OUT / "checkpoint_timing.csv", index=False)
display(timing_df)


## 6. Classification report and confusion matrix for the ensemble

In [ ]:

ensemble_preds = ensemble_probs.argmax(axis=1)
report = classification_report(
    holdout_labels, ensemble_preds, labels=np.arange(len(classes)),
    target_names=classes, zero_division=0, output_dict=False
)
print(report)
(OUT / "ensemble_classification_report.txt").write_text(report)

cm = confusion_matrix(holdout_labels, ensemble_preds, labels=np.arange(len(classes)))
fig, ax = plt.subplots(figsize=(20, 18))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes).plot(
    ax=ax, xticks_rotation=90, colorbar=False, values_format="d"
)
ax.set_title("Equal-weight ensemble — holdout confusion matrix")
plt.tight_layout()
plt.savefig(OUT / "ensemble_holdout_confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.show()


## 7. Save holdout predictions and summary

In [ ]:

pred_df = holdout_df[["image_id", "animal_class"]].copy()
pred_df["true_label"] = holdout_labels
pred_df["predicted_label"] = ensemble_preds
pred_df["predicted_class"] = [idx_to_class[int(i)] for i in ensemble_preds]
pred_df["correct"] = pred_df["true_label"] == pred_df["predicted_class"]
pred_df.to_csv(OUT / "ensemble_holdout_predictions.csv", index=False)

summary = {
    "holdout_rows": int(len(holdout_df)),
    "num_classes": int(len(classes)),
    "models_evaluated": list(model_probabilities.keys()),
    "ensemble": "equal-weight mean of model-family probabilities; each family averages available fold checkpoints",
    "metrics": metrics_df.to_dict(orient="records"),
    "checkpoint_paths": {
        cfg["name"]: [str(p) for p in cfg.get("checkpoints", [])]
        for cfg in MODEL_CONFIGS
    },
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2))
print("Saved evaluation artifacts to:", OUT)
print("\\nFiles:")
for p in sorted(OUT.iterdir()):
    print(" ", p.name, f"({p.stat().st_size/1024:.1f} KB)")
